In [5]:
from embedder import Embedder

embed = Embedder()

2026-06-29 10:39:48.294856855 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Q1. Embedding a query

In [6]:
query = "How does approximate nearest neighbor search work?"

embedding = embed.encode(query)

print(len(embedding))
print(f"First element of the embedding: {embedding[0]}")

384
First element of the embedding: -0.02058203437252893


Load data

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Q2. Cosine similarity

In [8]:
import numpy as np
target_doc = next(
    doc for doc in documents
    if "07-sqlitesearch-vector.md" in doc["filename"]
)
text = target_doc["content"]
doc_vec = embed.encode(text)
similarity = embedding.dot(doc_vec)
print(similarity)

0.36107027225589694


Q3. Chunking and search by hand

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
texts = [c["content"] for c in chunks]
filenames = [c["filename"] for c in chunks]

X = embed.encode_batch(texts)

scores = X.dot(embedding)

best_idx = np.argmax(scores)

best_filename = filenames[best_idx]
print(best_filename)

02-vector-search/lessons/07-sqlitesearch-vector.md


Q4. Vector search with minsearch

In [11]:
from minsearch import VectorSearch

In [12]:
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

In [13]:
query_Q4 = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query_Q4)
results = vindex.search(query_vector)
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


Q5. Text search vs vector search

In [14]:
from minsearch import Index

import inspect
print(inspect.signature(Index))

(text_fields, keyword_fields=None, numeric_fields=None, date_fields=None, vectorizer_params=None)


In [15]:
text_index = Index(text_fields=["content"])
text_index.fit(chunks)

In [28]:
query_Q5 = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(query_Q5)
text_files = list(dict.fromkeys(
    r["filename"] for r in text_results
))

print(text_files[:5])

['02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/03-embeddings-dataset.md', '02-vector-search/lessons/07-sqlitesearch-vector.md']


In [29]:

query_vector_Q5 = embed.encode(query_Q5)
vector_results = vindex.search(query_vector_Q5)
vector_files = list(dict.fromkeys(
    r["filename"] for r in vector_results
))

print(vector_files[:5])

['02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md', '05-monitoring/lessons/05-database.md', '02-vector-search/lessons/02-embeddings.md', '02-vector-search/lessons/04-vector-search.md']


In [30]:
set(vector_files[:5]) - set(text_files[:5])

{'02-vector-search/lessons/04-vector-search.md',
 '02-vector-search/lessons/08-pgvector.md',
 '05-monitoring/lessons/05-database.md'}

Q6. Hybrid search

In [31]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [32]:
query_Q6 = "How do I give the model access to tools?"

text_results_Q6 = text_index.search(query_Q6)
text_files = list(dict.fromkeys(
    r["filename"] for r in text_results_Q6
))

query_vector_Q6 = embed.encode(query_Q6)
vector_results_Q6 = vindex.search(query_vector_Q6)
vector_files = list(dict.fromkeys(
    r["filename"] for r in vector_results_Q6
))

results = rrf([vector_results_Q6, text_results_Q6])
print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
